# VitalyX — Road Safety Intelligence MVP

Single-cell Jupyter prototype. **Demo data is fictional; risk scores are illustrative.** Run the code cell below.

In [ ]:
# VitalyX — Road Safety Intelligence MVP
# Single-cell application for JupyterLab.
# Demo data is fictional. Risk scores are illustrative, not validated safety advice.

import os, sys, json, uuid, math, re, warnings, subprocess, importlib.util
from datetime import datetime, timezone
from pathlib import Path

def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        name = pip_name or import_name
        print(f"Installing missing dependency: {name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", name])

for _imp, _pip in [
    ("pandas", "pandas"),
    ("folium", "folium"),
    ("ipywidgets", "ipywidgets"),
    ("PIL", "Pillow"),
]:
    ensure_package(_imp, _pip)

import pandas as pd
import folium
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from PIL import Image

APP_NAME = "VitalyX"
DATA_FILE = Path("vitalyx_reports.csv")
IMAGE_DIR = Path("vitalyx_uploads")
IMAGE_DIR.mkdir(exist_ok=True)

# ---------- Data persistence ----------
COLUMNS = [
    "id", "area", "latitude", "longitude", "hazard", "description",
    "severity", "accidents", "rain", "reported_at", "status",
    "department", "contact", "verified_repair", "image_path",
    "review_notes", "classification_method"
]

def seed_demo_data():
    now = datetime.now(timezone.utc)
    sample = [
        dict(id="VX-DEMO-001", area="MG Road (demo)", latitude=26.8467,
             longitude=80.9462, hazard="Pothole",
             description="Large pothole in the road near a junction",
             severity=5, accidents=3, rain=True,
             reported_at=(now - pd.Timedelta(days=7)).isoformat(),
             status="Open", department="Municipal Roads (demo)",
             contact="Not verified", verified_repair=False,
             image_path="", review_notes="", classification_method="Demo label"),
        dict(id="VX-DEMO-002", area="University Road (demo)", latitude=26.9124,
             longitude=80.9431, hazard="Broken streetlight",
             description="Streetlight not working at night",
             severity=3, accidents=1, rain=False,
             reported_at=(now - pd.Timedelta(days=3)).isoformat(),
             status="Open", department="Street Lighting (demo)",
             contact="Not verified", verified_repair=False,
             image_path="", review_notes="", classification_method="Demo label"),
        dict(id="VX-DEMO-003", area="Market Lane (demo)", latitude=26.7606,
             longitude=83.3732, hazard="Blocked drain",
             description="Blocked drain with standing rainwater",
             severity=4, accidents=2, rain=True,
             reported_at=(now - pd.Timedelta(days=5)).isoformat(),
             status="Open", department="Drainage Department (demo)",
             contact="Not verified", verified_repair=False,
             image_path="", review_notes="", classification_method="Demo label"),
        dict(id="VX-DEMO-004", area="Residential Crossing (demo)", latitude=26.8500,
             longitude=80.9500, hazard="Garbage accumulation",
             description="Garbage pile partially obstructing the road",
             severity=2, accidents=0, rain=False,
             reported_at=(now - pd.Timedelta(days=1)).isoformat(),
             status="Open", department="Sanitation Department (demo)",
             contact="Not verified", verified_repair=False,
             image_path="", review_notes="", classification_method="Demo label"),
    ]
    return pd.DataFrame(sample, columns=COLUMNS)

if DATA_FILE.exists():
    try:
        reports = pd.read_csv(DATA_FILE)
        for col in COLUMNS:
            if col not in reports.columns:
                reports[col] = None
        reports = reports[COLUMNS]
        print(f"Loaded {len(reports)} saved report(s) from {DATA_FILE}.")
    except Exception as e:
        print("Could not read saved CSV; starting with demo records.", e)
        reports = seed_demo_data()
else:
    reports = seed_demo_data()

def persist():
    reports.to_csv(DATA_FILE, index=False)

HAZARD_RULES = {
    "Pothole": ["pothole", "hole in road", "road crater", "road hole"],
    "Broken streetlight": ["streetlight", "street light", "dark road", "lamp broken", "light not working"],
    "Blocked drain": ["blocked drain", "drain blocked", "waterlogging", "flooded road", "standing water"],
    "Garbage accumulation": ["garbage", "rubbish", "waste pile", "trash"],
    "Damaged road": ["cracked road", "road damage", "broken pavement", "damaged road"],
}
SEVERITY_DEFAULTS = {
    "Pothole": 4, "Broken streetlight": 3, "Blocked drain": 4,
    "Garbage accumulation": 2, "Damaged road": 4, "Other": 2
}

def classify_text(description):
    text = str(description).lower()
    for hazard, keywords in HAZARD_RULES.items():
        if any(word in text for word in keywords):
            return hazard, "Keyword baseline (not an ML model)"
    return "Other", "Fallback: manual review required"

# Optional genuine image model: Hugging Face zero-shot image classification.
# It is only used if transformers/torch are already installed and model loading succeeds.
# No automatic internet download is attempted. User can configure a local model separately.
VISION_PIPELINE = None
VISION_STATUS = "Not configured; image upload is stored for human review."
VISION_MODEL_ID = os.environ.get("VITALYX_VISION_MODEL", "").strip()

def try_load_vision_model():
    global VISION_PIPELINE, VISION_STATUS
    if not VISION_MODEL_ID:
        VISION_STATUS = "Not configured. Text keyword baseline is used; uploaded images need human review."
        return
    try:
        if importlib.util.find_spec("transformers") is None or importlib.util.find_spec("torch") is None:
            VISION_STATUS = "Model ID configured but transformers/torch are not installed."
            return
        from transformers import pipeline
        VISION_PIPELINE = pipeline(
            "zero-shot-image-classification",
            model=VISION_MODEL_ID,
            device=-1
        )
        VISION_STATUS = f"Vision model loaded: {VISION_MODEL_ID}"
    except Exception as e:
        VISION_PIPELINE = None
        VISION_STATUS = f"Vision model unavailable ({type(e).__name__}); manual review required."

def classify_image(image_path):
    if not image_path or not Path(image_path).exists():
        return None, None, "No image provided"
    if VISION_PIPELINE is None:
        return None, None, "Image stored; no vision model configured, manual review required"
    labels = [
        "a pothole in a road", "a damaged or cracked road",
        "a flooded road or waterlogging", "a blocked roadside drain",
        "garbage dumped on a road", "a broken streetlight at night",
        "a normal safe road with no visible hazard"
    ]
    try:
        result = VISION_PIPELINE(Image.open(image_path).convert("RGB"), candidate_labels=labels)
        best = result[0]
        label = best["label"].lower()
        score = float(best["score"])
        if "pothole" in label:
            hazard = "Pothole"
        elif "damaged" in label or "cracked" in label:
            hazard = "Damaged road"
        elif "flooded" in label or "waterlogging" in label:
            hazard = "Blocked drain"
        elif "drain" in label:
            hazard = "Blocked drain"
        elif "garbage" in label:
            hazard = "Garbage accumulation"
        elif "streetlight" in label:
            hazard = "Broken streetlight"
        else:
            hazard = "Other"
        return hazard, score, f"Vision model: {VISION_MODEL_ID}"
    except Exception as e:
        return None, None, f"Vision inference failed ({type(e).__name__}); manual review required"

try_load_vision_model()

def risk_score(row, now=None):
    if str(row.get("status", "")) == "Verified Fixed" and bool(row.get("verified_repair", False)):
        return 5
    now = now or datetime.now(timezone.utc)
    try:
        reported = pd.to_datetime(row.get("reported_at"), utc=True)
        age_days = max(0.0, (now - reported).total_seconds() / 86400)
    except Exception:
        age_days = 0.0
    try:
        severity = max(1, min(5, int(float(row.get("severity", 2))))) / 5 * 40
    except Exception:
        severity = 16
    try:
        accidents = max(0, int(float(row.get("accidents", 0))))
    except Exception:
        accidents = 0
    accident_score = min(accidents, 5) / 5 * 25
    rain_score = 15 if str(row.get("rain", False)).lower() in ("true", "1", "yes") else 0
    delay_score = min(age_days / 14, 1) * 20
    score = severity + accident_score + rain_score + delay_score
    if str(row.get("status", "")) == "Under Review":
        score *= 0.95
    return int(max(1, min(100, round(score))))

def risk_level(score):
    if score >= 80: return "Very High"
    if score >= 50: return "High"
    if score >= 25: return "Moderate"
    return "Low"

def refresh_scores():
    global reports
    if reports.empty:
        reports["risk_score"] = pd.Series(dtype=int)
        reports["risk_level"] = pd.Series(dtype=str)
    else:
        reports["risk_score"] = reports.apply(risk_score, axis=1)
        reports["risk_level"] = reports["risk_score"].apply(risk_level)

def add_report(area, description, lat, lon, severity, accidents, rain, image_path=""):
    global reports
    hazard, method = classify_text(description)
    img_hazard, confidence, image_method = classify_image(image_path)
    if img_hazard and confidence is not None and confidence >= 0.50:
        hazard = img_hazard
        method = f"{image_method}; confidence={confidence:.2f} (review recommended)"
    elif image_path:
        method = f"{method}; {image_method}"
    row = {
        "id": "VX-" + uuid.uuid4().hex[:8].upper(),
        "area": area.strip(),
        "latitude": float(lat),
        "longitude": float(lon),
        "hazard": hazard,
        "description": description.strip(),
        "severity": int(severity),
        "accidents": int(accidents),
        "rain": bool(rain),
        "reported_at": datetime.now(timezone.utc).isoformat(),
        "status": "Open",
        "department": "Unverified — assignment required",
        "contact": "Not verified",
        "verified_repair": False,
        "image_path": image_path or "",
        "review_notes": "",
        "classification_method": method,
    }
    reports = pd.concat([reports, pd.DataFrame([row])], ignore_index=True)
    refresh_scores()
    persist()
    return row

def dashboard():
    refresh_scores()
    clear_output(wait=True)
    display(HTML(f"""
    <div style="padding:16px;border-radius:12px;background:#111827;color:white">
      <h1 style="margin:0">🛣️ {APP_NAME}</h1>
      <p style="margin:6px 0 0">Neighbourhood Road Safety Intelligence · <b>DEMO / prototype</b></p>
      <p style="margin:6px 0 0;font-size:12px">{VISION_STATUS}</p>
    </div>
    """))
    total = len(reports)
    active = int(reports["status"].isin(["Open", "Under Review"]).sum()) if total else 0
    high = int((reports["risk_score"] >= 50).sum()) if total else 0
    fixed = int((reports["status"] == "Verified Fixed").sum()) if total else 0
    display(widgets.HBox([
        widgets.HTML(f"<div style='padding:12px;border:1px solid #ddd;border-radius:8px'><b>Reports</b><h2>{total}</h2></div>"),
        widgets.HTML(f"<div style='padding:12px;border:1px solid #ddd;border-radius:8px'><b>Open</b><h2>{active}</h2></div>"),
        widgets.HTML(f"<div style='padding:12px;border:1px solid #ddd;border-radius:8px'><b>Risk ≥ 50</b><h2>{high}</h2></div>"),
        widgets.HTML(f"<div style='padding:12px;border:1px solid #ddd;border-radius:8px'><b>Verified fixed</b><h2>{fixed}</h2></div>")
    ]))
    if total:
        display(reports.sort_values("risk_score", ascending=False)[
            ["id", "area", "hazard", "risk_score", "risk_level", "status", "classification_method"]
        ].reset_index(drop=True))
        center = [reports["latitude"].astype(float).mean(), reports["longitude"].astype(float).mean()]
    else:
        center = [26.85, 80.95]
    road_map = folium.Map(location=center, zoom_start=11)
    color = {"Very High":"red", "High":"orange", "Moderate":"blue", "Low":"green"}
    for _, r in reports.iterrows():
        popup = (
            f"<b>{r['area']}</b><br>Hazard: {r['hazard']}<br>"
            f"Risk: {r['risk_score']}/100 ({r['risk_level']})<br>"
            f"Status: {r['status']}<br>Department: {r['department']}<br>"
            f"Contact: {r['contact']}<br><i>Prototype data; verify before relying on it.</i>"
        )
        folium.CircleMarker(
            [float(r["latitude"]), float(r["longitude"])],
            radius=8, color=color.get(r["risk_level"], "gray"),
            fill=True, fill_opacity=0.8,
            tooltip=f"{r['hazard']} · {r['risk_score']}/100",
            popup=folium.Popup(popup, max_width=320)
        ).add_to(road_map)
    display(road_map)
    display(HTML("<p><b>Risk score note:</b> Illustrative, rule-based score; not calibrated against real accident data and not a guarantee that a road is safe.</p>"))

# ---------- Report form ----------
area_w = widgets.Text(description="Area", placeholder="e.g. Junction near college", layout=widgets.Layout(width="500px"))
desc_w = widgets.Textarea(description="Problem", placeholder="Describe what you observed", layout=widgets.Layout(width="500px", height="80px"))
lat_w = widgets.FloatText(description="Latitude", value=26.8467)
lon_w = widgets.FloatText(description="Longitude", value=80.9462)
sev_w = widgets.IntSlider(description="Severity", min=1, max=5, value=3, continuous_update=False)
acc_w = widgets.IntSlider(description="Past accidents", min=0, max=5, value=0, continuous_update=False)
rain_w = widgets.Checkbox(description="Rain / waterlogging conditions", value=False)
image_w = widgets.FileUpload(description="Upload photo", accept="image/*", multiple=False)
submit_w = widgets.Button(description="Submit report", button_style="primary", icon="check")
submit_out = widgets.Output()

def on_submit(_):
    with submit_out:
        clear_output()
        area, desc = area_w.value.strip(), desc_w.value.strip()
        lat, lon = lat_w.value, lon_w.value
        if not area or not desc:
            print("Enter an area and a description.")
            return
        if not (-90 <= lat <= 90 and -180 <= lon <= 180):
            print("Invalid latitude/longitude.")
            return
        saved_image = ""
        try:
            value = image_w.value
            if value:
                if isinstance(value, dict):  # ipywidgets 8
                    item = next(iter(value.values()))
                    content = item["content"]
                    filename = item.get("name", "upload.jpg")
                else:  # ipywidgets 7
                    item = value[0]
                    content = item["content"]
                    filename = item.get("name", "upload.jpg")
                ext = Path(filename).suffix.lower()
                if ext not in [".jpg", ".jpeg", ".png", ".webp"]:
                    ext = ".jpg"
                saved_image = str(IMAGE_DIR / f"{uuid.uuid4().hex}{ext}")
                with open(saved_image, "wb") as f:
                    f.write(bytes(content))
                # Validate file can be opened as an image.
                with Image.open(saved_image) as im:
                    im.verify()
        except Exception as e:
            saved_image = ""
            print(f"Image could not be stored/validated: {type(e).__name__}. Continuing without image.")
        row = add_report(area, desc, lat, lon, sev_w.value, acc_w.value, rain_w.value, saved_image)
        print("Report saved:", row["id"])
        print("Detected category:", row["hazard"])
        print("Classification:", row["classification_method"])
        print("Risk score:", int(reports.loc[reports["id"] == row["id"], "risk_score"].iloc[0]))
        print("Department/contact fields remain unverified.")
        print("Re-run the dashboard display at the end of this cell to refresh the map/table.")

submit_w.on_click(on_submit)

# ---------- Search ----------
search_w = widgets.Text(description="Search", placeholder="Area, hazard, description or status", layout=widgets.Layout(width="500px"))
search_btn = widgets.Button(description="Search reports")
search_out = widgets.Output()
def on_search(_):
    with search_out:
        clear_output()
        q = search_w.value.strip().lower()
        if not q:
            print("Enter a search term.")
            return
        mask = reports.apply(lambda row: row.astype(str).str.lower().str.contains(re.escape(q), na=False).any(), axis=1)
        result = reports[mask].copy()
        if result.empty:
            print("No matching reports.")
        else:
            display(result.sort_values("risk_score", ascending=False)[
                ["id", "area", "hazard", "risk_score", "risk_level", "status", "department", "contact"]
            ].reset_index(drop=True))
search_btn.on_click(on_search)

# ---------- Repair review ----------
repair_select = widgets.Dropdown(description="Report", options=[], layout=widgets.Layout(width="600px"))
repair_notes_w = widgets.Textarea(description="Evidence", placeholder="Inspection notes / evidence reviewed", layout=widgets.Layout(width="600px", height="80px"))
repair_confirm = widgets.Checkbox(description="Human reviewer has checked repair evidence", value=False)
repair_btn = widgets.Button(description="Mark verified fixed", button_style="success")
repair_out = widgets.Output()

def refresh_repair_options():
    repair_select.options = [
        (f"{r['id']} — {r['area']} — {r['status']}", r["id"])
        for _, r in reports.iterrows() if str(r["status"]) != "Verified Fixed"
    ]

def on_repair(_):
    with repair_out:
        clear_output()
        rid = repair_select.value
        notes = repair_notes_w.value.strip()
        if not rid:
            print("No open report selected.")
            return
        if not repair_confirm.value or not notes:
            print("Review the evidence, tick the confirmation box, and enter notes.")
            return
        idxs = reports.index[reports["id"] == rid]
        if len(idxs) == 0:
            print("Report not found.")
            return
        idx = idxs[0]
        reports.loc[idx, "status"] = "Verified Fixed"
        reports.loc[idx, "verified_repair"] = True
        reports.loc[idx, "review_notes"] = notes
        refresh_scores()
        persist()
        print(f"{rid} marked Verified Fixed by a human reviewer.")
        print("Illustrative score now:", int(reports.loc[idx, "risk_score"]))
        print("This notebook does not send live notifications.")
        refresh_repair_options()

repair_btn.on_click(on_repair)
refresh_repair_options()

# ---------- Export ----------
export_btn = widgets.Button(description="Export CSV", icon="download")
export_out = widgets.Output()
def on_export(_):
    with export_out:
        clear_output()
        refresh_scores()
        persist()
        print(f"Saved report data to: {DATA_FILE.resolve()}")
        print(f"Reports exported: {len(reports)}")
export_btn.on_click(on_export)

# ---------- Display app ----------
display(HTML("<h2>1. Safety dashboard and map</h2>"))
dashboard()
display(HTML("<hr><h2>2. Submit a road hazard report</h2><p>Image classification requires a configured vision model; otherwise the photo is stored for review.</p>"))
display(widgets.VBox([area_w, desc_w, lat_w, lon_w, sev_w, acc_w, rain_w, image_w, submit_w, submit_out]))
display(HTML("<hr><h2>3. Search reports</h2>"))
display(widgets.VBox([search_w, search_btn, search_out]))
display(HTML("<hr><h2>4. Repair verification</h2><p>Only mark a repair fixed after evidence is actually reviewed.</p>"))
display(widgets.VBox([repair_select, repair_notes_w, repair_confirm, repair_btn, repair_out]))
display(HTML("<hr><h2>5. Export data</h2>"))
display(widgets.VBox([export_btn, export_out]))

print("\nVitalyX is ready.")
print(f"Data persists in {DATA_FILE.resolve()}")
print(f"Uploaded images are stored under {IMAGE_DIR.resolve()}")
print("Demo records are fictional. No live municipal contacts, weather, accident feed, or notifications are connected.")
